# 02 · Exploratory data analysis (EDA)

**Goal.** Understand which variables separate LGG from GBM and which ones are related to age, and detect any problem in the data **before** training models.

The structure follows the course notebook `EDA_for_Extreme_OSA_Classification`, adapted to this dataset. Here there is **one continuous variable** (age) and **22 binary or nominal ones** (gender, race, 20 mutations), so the tests change:

| Variable type | Test | Effect size |
|---|---|---|
| Continuous (age) | Mann–Whitney U (Welch t-test for comparison) | Cohen's d, AUC = U/(n₁n₀) |
| Binary (gender, mutations) | Fisher's exact test (2×2 table) | Odds ratio, phi coefficient |
| Nominal with tiny groups (race) | Fisher's exact test, white vs non-white | Odds ratio |

With 23 tests at once, p-values are corrected for multiple comparisons (Bonferroni and Benjamini–Hochberg).

| # | Section | Main question |
|---|---|---|
| 1 | Setup | Data, colours, feature lists |
| 2 | Structure and quality | Is the clean dataset usable? Repeated profiles? Plausible ages? |
| 3 | Target and leakage | Are the classes balanced? Which columns cheat? |
| 4 | Univariate analysis per class | How do age and each mutation differ between classes? |
| 5 | Normality | Can a t-test be used for age? |
| 6 | Tests and effect size | Which differences are real, and how large? |
| 7 | Outliers | Are there extreme ages? |
| 8 | Correlation and multicollinearity | Are some features redundant? |
| 9 | Relevance for the target | Which features are the best single predictors? |
| 10 | Conditional probability | How does P(GBM) change along each feature? |
| 11 | Multivariate view | Do several features together separate the classes? |
| 12 | Quick baseline | How much signal is there? What does leakage look like? |
| 13 | EDA for the regression target | Which features are related to age? |
| 14 | Summary | Decisions for the modelling stages |

PCA, MDS and clustering of the mutation profiles are in notebook 03.

**Important.** All statistics here are computed on the full dataset to *describe* it. They are **not** used to select features: selecting features with the labels outside cross-validation would leak information (course slides, *Cross-validation: right and wrong*).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Project root (folder containing src/config.py) not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

from src import config, data, eda, plotting, reporting
from src.plotting import CLASS_COLORS, LGG_COLOR, GBM_COLOR, NEUTRAL

plotting.set_style()
pd.set_option("display.precision", 3)

df = data.load_processed()
df["Class"] = df["Grade"].map(config.CLASS_LABELS)
df["Race_nonwhite"] = (df["Race"] != 0).astype(int)
GENES = config.GENE_COLS
y = df["Grade"].values
results = {}          # everything that goes to results/logs/02_eda.json
print(df.shape)

## 2. Structure and quality

The cleaning was done and verified in notebook 01, so here we only confirm types and missing values, and add two checks:
- **Plausibility**: diffuse gliomas occur in adults and also in adolescents, so ages between roughly 10 and 90 are plausible; values outside would be errors.
- **Repeated profiles**: different patients can share exactly the same predictor values (for example, no mutation in any of the 20 genes). They are different patients, so this is not a duplicate to remove, but it limits how finely any model can separate them: patients with identical inputs always get the same prediction.

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Age range: {:.2f} - {:.2f} years".format(df["Age_at_diagnosis"].min(), df["Age_at_diagnosis"].max()))

n_mut = df[GENES].sum(axis=1)
profile_cols = ["Gender", "Race"] + GENES
n_profiles = df[profile_cols].drop_duplicates().shape[0]
no_mutation = int((n_mut == 0).sum())
print(f"Distinct (gender, race, mutation) profiles: {n_profiles} for {len(df)} patients")
print(f"Patients with no mutation in the 20 genes: {no_mutation} "
      f"({df.loc[n_mut == 0, 'Class'].value_counts().to_dict()})")

results["quality"] = {"missing_values": int(df.isna().sum().sum()),
                      "age_min": df["Age_at_diagnosis"].min(), "age_max": df["Age_at_diagnosis"].max(),
                      "distinct_profiles_without_age": n_profiles,
                      "patients_without_mutations": no_mutation,
                      "patients_without_mutations_by_class": df.loc[n_mut == 0, "Class"].value_counts().to_dict(),
                      "mutations_per_patient": n_mut.describe().round(2).to_dict(),
                      "patients_with_6_or_more_mutations": int((n_mut >= 6).sum())}

fig, ax = plt.subplots(figsize=(6.5, 3.2))
n_mut_grouped = n_mut.clip(upper=6).map(lambda k: "6+" if k == 6 else str(k))
counts = pd.crosstab(n_mut_grouped, df["Class"])[["LGG", "GBM"]].reindex([str(k) for k in range(6)] + ["6+"], fill_value=0)
print("Patients with 6 or more mutated genes:", int((n_mut >= 6).sum()), "- maximum:", int(n_mut.max()))
counts.plot.bar(ax=ax, color=[LGG_COLOR, GBM_COLOR], width=0.8, edgecolor="white", linewidth=1)
ax.set_xlabel("Number of mutated genes (out of 20)")
ax.set_ylabel("Patients")
ax.set_title("Number of mutated genes per patient")
ax.tick_params(axis="x", rotation=0)
ax.legend(title=None)
plotting.save_figure(fig, "eda_mutations_per_patient")
plt.show()

## 3. Target variable and data leakage

### 3.1 Class balance

The proportion of the majority class is the accuracy of a trivial model that always predicts it. Any classifier has to beat it, and because the classes are not perfectly balanced, accuracy alone can be misleading: we will also report AUC, sensitivity and specificity, and use stratified folds.

In [ ]:
balance = df["Class"].value_counts().reindex(["LGG", "GBM"]).rename("Patients").to_frame()
balance["Proportion"] = balance["Patients"] / len(df)
display(balance)
results["class_balance"] = balance.to_dict(orient="index")
results["majority_baseline_accuracy"] = float(balance["Proportion"].max())

fig, ax = plt.subplots(figsize=(4, 2.6))
ax.barh(balance.index, balance["Patients"], color=[LGG_COLOR, GBM_COLOR], height=0.6)
for i, (n, p) in enumerate(zip(balance["Patients"], balance["Proportion"])):
    ax.text(n + 8, i, f"{n} ({100 * p:.1f}%)", va="center", color=plotting.INK)
ax.set_xlim(0, balance["Patients"].max() * 1.3)
ax.set_xlabel("Patients")
ax.set_title("Class balance")
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
plotting.save_figure(fig, "eda_class_balance")
plt.show()
reporting.save_table_tex(balance.reset_index().rename(columns={"Class": "Grade"}), "eda_class_balance",
                         float_format="{:.3f}")

### 3.2 Data leakage: diagnosis and project

The grade is assigned from the histological diagnosis: in this dataset every glioblastoma is GBM and every other diagnosis is LGG. A model given `Primary_Diagnosis` would simply read the answer. (`Project` was already dropped in notebook 01 for the same reason.) Section 12 shows the fake performance this would produce.

In [ ]:
ct = pd.crosstab(df[config.MULTICLASS_TARGET], df["Class"])[["LGG", "GBM"]]
display(ct)
results["diagnosis_vs_grade"] = ct.to_dict(orient="index")

fig, ax = plt.subplots(figsize=(5.2, 3.4))
sns.heatmap(ct, annot=True, fmt="d", cmap=plotting.SEQUENTIAL, cbar=False, linewidths=2, linecolor="white", ax=ax)
ax.set_xlabel("Grade")
ax.set_ylabel("")
ax.set_title("Primary diagnosis determines the grade")
plotting.save_figure(fig, "eda_diagnosis_vs_grade")
plt.show()

## 4. Univariate analysis per class

### 4.1 Age: descriptive statistics, distributions and ECDF

For classification, what matters is the **difference between classes**. The histogram shows the shape of each distribution, the box/violin plot the shift of the median and quartiles, and the ECDF (fraction of patients below each age) shows the best single age threshold as the point of maximum vertical distance between the two curves (the Kolmogorov–Smirnov statistic).

In [ ]:
age_desc = eda.describe_by_class(df, "Age_at_diagnosis")
display(age_desc)
results["age_by_class"] = age_desc.to_dict(orient="index")
reporting.save_table_tex(age_desc.reset_index().rename(columns={"Class": "Grade", "count": "n", "std": "SD"}),
                         "eda_age_by_grade", float_format="{:.1f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
bins = np.arange(10, 95, 5)
for code_, label in config.CLASS_LABELS.items():
    x = df.loc[df["Grade"] == code_, "Age_at_diagnosis"]
    axes[0].hist(x, bins=bins, alpha=0.55, color=CLASS_COLORS[label], label=label, edgecolor="white")
    xs = np.sort(x)
    axes[2].step(xs, np.arange(1, len(xs) + 1) / len(xs), where="post", color=CLASS_COLORS[label], label=label)
axes[0].set_xlabel("Age at diagnosis (years)"); axes[0].set_ylabel("Patients")
axes[0].set_title("Histogram"); axes[0].legend()

sns.violinplot(data=df, x="Class", y="Age_at_diagnosis", hue="Class", order=["LGG", "GBM"],
               palette=CLASS_COLORS, inner=None, linewidth=0, ax=axes[1], legend=False)
for coll in axes[1].collections:
    coll.set_alpha(0.35)
sns.boxplot(data=df, x="Class", y="Age_at_diagnosis", order=["LGG", "GBM"], width=0.25, color="white",
            linewidth=1.2, fliersize=3, ax=axes[1])
axes[1].set_xlabel(""); axes[1].set_ylabel("Age at diagnosis (years)"); axes[1].set_title("Box and violin plot")

ks = stats.ks_2samp(df.loc[y == 1, "Age_at_diagnosis"], df.loc[y == 0, "Age_at_diagnosis"])
axes[2].set_xlabel("Age at diagnosis (years)"); axes[2].set_ylabel("Cumulative proportion")
axes[2].set_title(f"ECDF (KS statistic = {ks.statistic:.2f})"); axes[2].legend()
fig.tight_layout()
plotting.save_figure(fig, "eda_age_by_grade")
plt.show()
results["age_ks"] = {"statistic": ks.statistic, "p_value": ks.pvalue}

### 4.2 Mutation prevalence per class

For a binary feature, its whole distribution in each class is a single number: the percentage of patients with the mutation. Genes are sorted by the difference between classes.

In [ ]:
prev = df.groupby("Class")[GENES].mean().T[["LGG", "GBM"]]
prev["Difference (GBM - LGG)"] = prev["GBM"] - prev["LGG"]
prev = prev.sort_values("Difference (GBM - LGG)")
display((100 * prev).round(1))

fig, ax = plt.subplots(figsize=(6.5, 6))
pos = np.arange(len(prev))
ax.barh(pos + 0.2, 100 * prev["LGG"], height=0.38, color=LGG_COLOR, label="LGG")
ax.barh(pos - 0.2, 100 * prev["GBM"], height=0.38, color=GBM_COLOR, label="GBM")
ax.set_yticks(pos, prev.index)
ax.set_xlabel("Patients with the mutation (%)")
ax.set_title("Mutation prevalence by grade")
ax.legend(loc="upper right")
ax.grid(axis="y", visible=False)
plotting.save_figure(fig, "eda_mutation_prevalence")
plt.show()

gender_race = pd.concat([pd.crosstab(df["Gender"].map({0: "Male", 1: "Female"}), df["Class"], normalize="columns"),
                         pd.crosstab(df["Race"].map({0: "White", 1: "Black or African American", 2: "Asian",
                                                     3: "American Indian or Alaska Native"}), df["Class"],
                                     normalize="columns")])[["LGG", "GBM"]]
display((100 * gender_race).round(1))
results["gender_race_by_class_percent"] = (100 * gender_race).round(2).to_dict(orient="index")

## 5. Normality check for age

The t-test assumes approximately normal data in each class. We check it with the Shapiro–Wilk test (small p-value = not normal) and Q–Q plots (points on the line = normal). With several hundred patients per class Shapiro–Wilk detects even small deviations, so the plot matters as much as the p-value.

In [ ]:
shapiro = eda.shapiro_by_class(df, "Age_at_diagnosis")
display(shapiro)
results["age_shapiro"] = shapiro.to_dict(orient="records")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
for ax, (code_, label) in zip(axes, config.CLASS_LABELS.items()):
    (osm, osr), (slope, intercept, _) = stats.probplot(df.loc[df["Grade"] == code_, "Age_at_diagnosis"])
    ax.plot(osm, osr, "o", ms=3, color=CLASS_COLORS[label], alpha=0.7)
    ax.plot(osm, slope * osm + intercept, color=NEUTRAL, lw=1.2)
    ax.set_title(f"{label}: Q-Q plot of age")
    ax.set_xlabel("Theoretical normal quantiles"); ax.set_ylabel("Age (years)")
fig.tight_layout()
plotting.save_figure(fig, "eda_age_qq")
plt.show()

## 6. Statistical tests and effect size

A p-value says **whether** a difference is unlikely to be chance; it does not say **how large** it is. For classification the size matters, so every test is reported with an effect size:
- **AUC** (for every feature): probability that a random GBM patient has a higher value than a random LGG patient. 0.5 = no separation, 1 = perfect. For age it equals U/(n₁n₀) from the Mann–Whitney test.
- **Cohen's d** for age (0.2 small, 0.5 medium, 0.8 large).
- **Odds ratio** for binary features (> 1: more frequent in GBM; < 1: more frequent in LGG). A +0.5 correction is added to every cell so that genes with zero mutated patients in one class still give a finite value.

**Multiple comparisons.** With 23 tests, the chance of at least one false positive at α = 0.05 is high. Bonferroni (multiply by 23) controls the probability of any false positive and is conservative; Benjamini–Hochberg controls the expected proportion of false discoveries and is less conservative. We report both.

In [ ]:
tests = eda.classification_tests(df, GENES)
cols_show = ["Feature", "Test", "LGG", "GBM", "Effect size", "AUC", "p-value", "p Bonferroni", "p BH"]
display(tests[cols_show].style.format({"AUC": "{:.3f}", "p-value": "{:.1e}", "p Bonferroni": "{:.1e}", "p BH": "{:.1e}"}))

n_sig_bonf = int((tests["p Bonferroni"] < 0.05).sum())
n_sig_bh = int((tests["p BH"] < 0.05).sum())
print(f"Significant at 0.05: {n_sig_bonf} with Bonferroni, {n_sig_bh} with Benjamini-Hochberg (out of {len(tests)})")
results["classification_tests"] = tests.drop(columns=["LGG", "GBM", "Effect size"]).to_dict(orient="records")
results["n_significant_bonferroni"] = n_sig_bonf
results["n_significant_bh"] = n_sig_bh

tex = tests[cols_show].copy()
for c in ["p-value", "p Bonferroni", "p BH"]:
    tex[c] = tex[c].map(lambda p: f"{p:.1e}" if p >= 1e-300 else "<1e-300")
tex["AUC"] = tex["AUC"].map("{:.3f}".format)
reporting.save_table_tex(tex, "eda_tests_classification", column_format="llrrlrrrr")

## 7. Outliers in age

Tukey's rule (below Q1 − 1.5·IQR or above Q3 + 1.5·IQR), computed **per class**, because an age that is usual for one grade may be unusual for the other. Outliers here are real patients with uncommon ages, not measurement errors (all ages passed the plausibility check), so they are **kept**. Binary variables cannot have outliers in this sense; the equivalent issue is very rare mutations (Section 4.2).

In [ ]:
outliers = eda.iqr_outliers(df, "Age_at_diagnosis")
display(outliers)
results["age_outliers"] = outliers.to_dict(orient="records")

## 8. Correlation between features and multicollinearity

For two binary variables, the Pearson correlation is the **phi coefficient**; between age and a binary variable it is the point-biserial correlation. The matrix uses Spearman (rank-based, the same as Pearson for binary variables) so that age is treated robustly.

Redundant features do not hurt the accuracy of tree ensembles, but they make the coefficients of linear and logistic regression unstable. The **VIF** measures how well each feature is predicted by all the others together: VIF = 1/(1 − R²); values above 5–10 indicate problematic collinearity (course slides, Chapter 3).

In [ ]:
features = ["Age_at_diagnosis", "Gender", "Race_nonwhite"] + GENES
corr = df[features].corr(method="spearman")
order = ["Age_at_diagnosis", "Gender", "Race_nonwhite"] + list(prev.index[::-1])
fig, ax = plt.subplots(figsize=(9, 7.5))
mask_upper = np.triu(np.ones((len(order), len(order)), dtype=bool))
sns.heatmap(corr.loc[order, order], mask=mask_upper, cmap=plotting.DIVERGING, vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Spearman correlation", "shrink": 0.7}, ax=ax)
ax.set_title("Correlation between features")
plotting.save_figure(fig, "eda_feature_correlation")
plt.show()

top_corr = eda.top_correlations(corr, n=10)
display(top_corr)
reporting.save_table_tex(top_corr, "eda_top_correlations", float_format="{:.2f}")
results["top_correlations"] = top_corr.to_dict(orient="records")

race_dummies = pd.get_dummies(df["Race"].map({0: "White", 1: "Race_black", 2: "Race_asian", 3: "Race_native"}))
race_dummies = race_dummies.drop(columns="White").astype(int)
X_vif = pd.concat([df[["Age_at_diagnosis", "Gender"]], race_dummies, df[GENES]], axis=1)
vif = eda.vif_table(X_vif)
display(vif)
reporting.save_table_tex(vif.head(10), "eda_vif", float_format="{:.2f}")
results["vif"] = vif.to_dict(orient="records")
print("Maximum VIF:", round(vif["VIF"].max(), 2))

## 9. Relevance of each feature for the target

### 9.1 Single-feature AUC with bootstrap confidence intervals, and mutual information

The AUC of each feature used alone is the performance of the simplest possible classifier ("predict GBM if the feature is high", or low for features more frequent in LGG; the table shows the direction). The 95% interval comes from 2000 bootstrap resamples, stratified by class. Mutual information is a second, model-free measure that also captures non-linear relations (0 = independent).

In [ ]:
relevance = eda.feature_relevance(df, features)
display(relevance)
results["relevance"] = relevance.to_dict(orient="records")
reporting.save_table_tex(relevance.head(10), "eda_relevance", float_format="{:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
r = relevance.iloc[::-1]
colors = [GBM_COLOR if d == "higher in GBM" else LGG_COLOR for d in r["Direction"]]
ax.errorbar(r["AUC"], np.arange(len(r)), xerr=np.clip([r["AUC"] - r["AUC CI low"], r["AUC CI high"] - r["AUC"]], 0, None),
            fmt="none", ecolor=NEUTRAL, elinewidth=1.2, capsize=2)
ax.scatter(r["AUC"], np.arange(len(r)), c=colors, s=40, zorder=3)
ax.axvline(0.5, color=NEUTRAL, ls="--", lw=1)
ax.set_yticks(np.arange(len(r)), r["Feature"])
ax.set_xlabel("Single-feature AUC (95% bootstrap CI)")
ax.set_title("Relevance of each feature alone")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=GBM_COLOR, label="More frequent / higher in GBM"),
                   Line2D([], [], marker="o", ls="", color=LGG_COLOR, label="More frequent / higher in LGG")],
          loc="lower right")
ax.grid(axis="y", visible=False)
plotting.save_figure(fig, "eda_feature_relevance")
plt.show()

### 9.2 ROC curves of the most relevant features and the best age threshold

The ROC curve of a binary feature has a single corner point (its sensitivity and specificity). For age, the threshold that maximises Youden's index J = sensitivity + specificity − 1 gives a simple clinical rule. It is chosen and evaluated on the same data, so it is optimistic: the models in stage 5 must clearly beat it under cross-validation.

In [ ]:
from sklearn.metrics import roc_curve
top = relevance["Feature"].head(4).tolist()
fig, ax = plt.subplots(figsize=(4.8, 4.6))
# Feature colours avoid the two class colours; line styles give a second, colour-free encoding
roc_colors = [plotting.CATEGORICAL[2], plotting.CATEGORICAL[6], plotting.CATEGORICAL[4], plotting.CATEGORICAL[3]]
roc_styles = ["-", "--", "-.", ":"]
for i, f in enumerate(top):
    score = df[f] if relevance.set_index("Feature").loc[f, "Direction"] == "higher in GBM" else -df[f]
    fpr, tpr, _ = roc_curve(y, score)
    auc = relevance.set_index("Feature").loc[f, "AUC"]
    ax.plot(fpr, tpr, color=roc_colors[i], ls=roc_styles[i],
            label=f"{f} (AUC = {auc:.3f})", marker="o" if f != "Age_at_diagnosis" else None, ms=4)
ax.plot([0, 1], [0, 1], ls="--", color=NEUTRAL, lw=1, label="Chance (AUC = 0.5)")
ax.set_xlabel("False positive rate (1 - specificity)")
ax.set_ylabel("True positive rate (sensitivity)")
ax.set_title("ROC curve of the top features used alone")
ax.legend(loc="lower right", fontsize=8)
plotting.save_figure(fig, "eda_roc_top_features")
plt.show()

youden = eda.youden_threshold(df["Age_at_diagnosis"], y)
print("Best age threshold (predict GBM if age >= t):", youden)
results["age_youden"] = youden

## 10. Conditional probability of GBM

A classifier estimates P(GBM | features). Here it is estimated empirically, one feature at a time, with 95% Wilson intervals:
- **Age**: patients are split into 5 groups of equal size (quintiles). An increasing curve means a monotonic relation, easy for logistic regression; a curve with a jump suggests a threshold, natural for a tree.
- **Mutations**: P(GBM) with and without each mutation. The dashed line is the overall proportion of GBM (the prior).

In [ ]:
cp_age = eda.conditional_probability_by_quantile(df, "Age_at_diagnosis")
cp_genes = eda.conditional_probability_binary(df, GENES)
cp_genes = cp_genes.set_index("Feature").loc[prev.index[::-1]].rename_axis("Feature").reset_index()
display(cp_age)
results["conditional_probability_age"] = cp_age.to_dict(orient="records")
results["conditional_probability_genes"] = cp_genes.to_dict(orient="records")
prior = y.mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
ax = axes[0]
ax.errorbar(cp_age["Midpoint"], cp_age["P(GBM)"],
            yerr=np.clip([cp_age["P(GBM)"] - cp_age["CI low"], cp_age["CI high"] - cp_age["P(GBM)"]], 0, None),
            color=GBM_COLOR, marker="o", capsize=3)
ax.axhline(prior, color=NEUTRAL, ls="--", lw=1)
ax.text(cp_age["Midpoint"].min(), prior + 0.02, "overall P(GBM)", color=plotting.INK_SECONDARY, fontsize=8)
ax.set_ylim(0, 1)
ax.set_xlabel("Age at diagnosis (median of each quintile, years)")
ax.set_ylabel("P(GBM)")
ax.set_title("P(GBM) by age quintile")

ax = axes[1]
pos = np.arange(len(cp_genes))
for col, color, off, lab in (("mutated", GBM_COLOR, 0.15, "Mutated"), ("not mutated", NEUTRAL, -0.15, "Not mutated")):
    p = cp_genes[f"P(GBM | {col})"]
    ax.errorbar(p, pos + off, xerr=np.clip([p - cp_genes[f"CI low ({col})"], cp_genes[f"CI high ({col})"] - p], 0, None),
                fmt="o", color=color, ms=5, capsize=2, label=lab)
ax.axvline(prior, color=NEUTRAL, ls="--", lw=1)
ax.set_yticks(pos, cp_genes["Feature"])
ax.set_xlim(0, 1)
ax.set_xlabel("P(GBM)")
ax.set_title("P(GBM) with and without each mutation")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plotting.save_figure(fig, "eda_conditional_probability")
plt.show()

## 11. Multivariate view

### 11.1 Age and IDH1 together

The two most relevant features are plotted together (IDH1 with a small vertical jitter so that the points do not overlap). If the classes occupy different regions, a combination of the two features separates them better than either alone.

### 11.2 Linear discriminant analysis (LDA)

LDA finds the linear combination of all features that best separates the two classes (largest ratio of between-class to within-class variance). The histogram of the patients projected on that axis shows the best separation a **linear** model can achieve; the overlap region is where errors will happen. Here LDA is fitted on all patients only as a *visualisation*: its separation is optimistic and it is not a performance estimate.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(config.SEED)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
for code_, label in config.CLASS_LABELS.items():
    sub = df[df["Grade"] == code_]
    ax.scatter(sub["Age_at_diagnosis"], sub["IDH1"] + rng.uniform(-0.18, 0.18, len(sub)),
               s=12, alpha=0.6, color=CLASS_COLORS[label], label=label, edgecolor="white", linewidth=0.3)
ax.set_yticks([0, 1], ["IDH1 not mutated", "IDH1 mutated"])
ax.set_xlabel("Age at diagnosis (years)")
ax.set_title("Age and IDH1 status")
ax.legend(loc="center right")

X_lda = StandardScaler().fit_transform(pd.concat([df[["Age_at_diagnosis", "Gender"]], race_dummies, df[GENES]], axis=1))
lda = LinearDiscriminantAnalysis().fit(X_lda, y)
z = lda.transform(X_lda).ravel()
ax = axes[1]
bins_z = np.linspace(z.min(), z.max(), 40)
for code_, label in config.CLASS_LABELS.items():
    ax.hist(z[y == code_], bins=bins_z, alpha=0.55, color=CLASS_COLORS[label], label=label, edgecolor="white")
ax.set_xlabel("LDA projection")
ax.set_ylabel("Patients")
ax.set_title("All features projected on the LDA axis")
ax.legend()
fig.tight_layout()
plotting.save_figure(fig, "eda_multivariate")
plt.show()

lda_coef = pd.Series(lda.scalings_.ravel(), index=["Age_at_diagnosis", "Gender"] + list(race_dummies.columns) + GENES)
display(lda_coef.sort_values(key=np.abs, ascending=False).head(8).rename("LDA coefficient (standardised)"))
results["lda_top_coefficients"] = lda_coef.sort_values(key=np.abs, ascending=False).head(8).to_dict()

## 12. Quick baseline (sanity check of the EDA)

Not the modelling stage yet: a quick cross-validated logistic regression to check the conclusions above.
- **How much signal is there?** Compare a dummy model (always predicts the prior), single features and all features.
- **What does leakage look like?** Add the one-hot `Primary_Diagnosis` to the features.

Stratified 10-fold CV with seed 42 (same protocol as the models). Scaling and one-hot encoding are inside a `Pipeline`, so they are fitted on the training folds only. Metrics: ROC-AUC and balanced accuracy (mean of sensitivity and specificity, 0.5 for a trivial model).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from src.pipelines import make_preprocessor

df_leak = pd.concat([df, pd.get_dummies(df[config.MULTICLASS_TARGET], prefix="Dx").astype(int)], axis=1)
dx_cols = [c for c in df_leak.columns if c.startswith("Dx_")]
clf_features = data.feature_columns("classification")
feature_sets = {
    "Dummy (prior)": None,
    "Age only": ["Age_at_diagnosis"],
    "IDH1 only": ["IDH1"],
    "Age + IDH1": ["Age_at_diagnosis", "IDH1"],
    "20 genes only": GENES,
    "All 23 predictors": clf_features,
    "All 23 + diagnosis (LEAKAGE)": clf_features + dx_cols,
}
cv = StratifiedKFold(n_splits=config.N_OUTER_FOLDS, shuffle=True, random_state=config.SEED)
rows = []
for name, cols in feature_sets.items():
    if cols is None:
        model, X = DummyClassifier(strategy="prior"), df_leak[["Age_at_diagnosis"]]
    else:
        model = make_pipeline(make_preprocessor(cols), LogisticRegression(max_iter=5000))
        X = df_leak[cols]
    s = cross_validate(model, X, y, cv=cv, scoring=["roc_auc", "balanced_accuracy"])
    rows.append({"Feature set": name, "Predictors": 0 if cols is None else len(cols),
                 "ROC-AUC mean": s["test_roc_auc"].mean(), "ROC-AUC SD": s["test_roc_auc"].std(),
                 "Balanced acc. mean": s["test_balanced_accuracy"].mean(),
                 "Balanced acc. SD": s["test_balanced_accuracy"].std()})
baseline = pd.DataFrame(rows)
display(baseline)
reporting.save_table_tex(baseline, "eda_baseline", float_format="{:.3f}", column_format="lrrrrr")
results["baseline"] = baseline.to_dict(orient="records")

## 13. EDA for the regression target (age)

The regression task predicts age at diagnosis from gender, race and the 20 mutations, **without** using the grade. Two questions:
- How is age distributed? (Shape of the target, scale of the errors to expect: a model that always predicts the mean has an error of about one standard deviation.)
- Which features are related to age? Mann–Whitney test between patients with and without each mutation, with the difference of medians and the rank-biserial correlation as effect sizes, and Benjamini–Hochberg correction.

The grade is strongly related to age (Section 4) and also to several mutations. A regression model can therefore partly recover the grade through those mutations: this is not leakage (the grade is not an input), but it must be kept in mind when interpreting the regression coefficients.

In [ ]:
age = df["Age_at_diagnosis"]
results["age_overall"] = {**age.describe().round(3).to_dict(), "skewness": float(age.skew()),
                          "shapiro_p": float(stats.shapiro(age).pvalue)}
print(results["age_overall"])

age_tests = eda.age_by_binary(df, ["Gender"] + GENES)
display(age_tests.style.format({"Median age with": "{:.1f}", "Median age without": "{:.1f}",
                                "Median difference": "{:+.1f}", "Rank-biserial r": "{:+.3f}",
                                "p-value": "{:.1e}", "p Bonferroni": "{:.1e}", "p BH": "{:.1e}"}))
results["age_tests"] = age_tests.to_dict(orient="records")
results["n_age_significant_bh"] = int((age_tests["p BH"] < 0.05).sum())

race_groups = [g["Age_at_diagnosis"].values for _, g in df.groupby("Race")]
kw = stats.kruskal(*race_groups)
results["age_by_race_kruskal"] = {"H": kw.statistic, "p_value": kw.pvalue,
                                  "median_by_race": df.groupby("Race")["Age_at_diagnosis"].median().to_dict()}
print("Age by race (Kruskal-Wallis): H = {:.2f}, p = {:.3g}".format(kw.statistic, kw.pvalue))

tex = age_tests.copy()
tex["Median age with"] = tex["Median age with"].map("{:.1f}".format)
tex["Median age without"] = tex["Median age without"].map("{:.1f}".format)
tex["Median difference"] = tex["Median difference"].map("{:+.1f}".format)
tex["Rank-biserial r"] = tex["Rank-biserial r"].map("{:+.3f}".format)
for c in ["p-value", "p Bonferroni", "p BH"]:
    tex[c] = tex[c].map(lambda p: f"{p:.1e}" if p >= 1e-300 else "<1e-300")
reporting.save_table_tex(tex, "eda_tests_regression", column_format="lrrrrrrrr")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.5]})
ax = axes[0]
ax.hist(age, bins=np.arange(10, 95, 5), color=plotting.CATEGORICAL[2], edgecolor="white", alpha=0.85)
ax.axvline(age.mean(), color=plotting.INK, lw=1.2, ls="--")
ax.text(age.mean() + 1, ax.get_ylim()[1] * 0.92, f"mean {age.mean():.1f}", fontsize=8)
ax.set_xlabel("Age at diagnosis (years)"); ax.set_ylabel("Patients")
ax.set_title("Distribution of the regression target")

ax = axes[1]
top_age = age_tests.head(8)["Feature"].tolist()
long = pd.concat([pd.DataFrame({"Feature": f, "Status": df[f].map({0: "Without", 1: "With"}), "Age": age})
                  for f in top_age])
sns.boxplot(data=long, y="Feature", x="Age", hue="Status", hue_order=["Without", "With"], order=top_age,
            palette={"Without": "#c3c2b7", "With": plotting.CATEGORICAL[2]}, fliersize=2, linewidth=1, ax=ax)
ax.set_xlabel("Age at diagnosis (years)"); ax.set_ylabel("")
ax.set_title("Age with and without each feature (8 strongest)")
ax.legend(title=None, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
fig.tight_layout()
plotting.save_figure(fig, "eda_age_regression_target")
plt.show()

## 14. Summary

The table below is generated from the results above (no number is typed by hand). It is saved with all intermediate results in `results/logs/02_eda.json`, and the decisions it supports are recorded in `docs/DECISIONS.md`.

In [ ]:
t = tests.set_index("Feature")
rel = relevance.set_index("Feature")
summary_rows = [
    ("Class balance", f"{balance.loc['LGG', 'Patients']} LGG / {balance.loc['GBM', 'Patients']} GBM; "
                      f"majority baseline accuracy {results['majority_baseline_accuracy']:.3f}"),
    ("Leakage", f"Primary_Diagnosis determines Grade; with it, CV ROC-AUC = "
                f"{baseline.set_index('Feature set').loc['All 23 + diagnosis (LEAKAGE)', 'ROC-AUC mean']:.3f}"),
    ("Age", f"median {age_desc.loc['LGG', '50%']:.1f} (LGG) vs {age_desc.loc['GBM', '50%']:.1f} (GBM); "
            f"Cohen d = {t.loc['Age_at_diagnosis', 'Cohen d']:.2f}; AUC = {rel.loc['Age_at_diagnosis', 'AUC']:.3f}; "
            f"not normal (Shapiro p < 0.05 in both classes: {bool((shapiro['p-value'] < 0.05).all())})"),
    ("Most relevant features", ", ".join(f"{f} (AUC {rel.loc[f, 'AUC']:.3f})" for f in relevance['Feature'].head(5))),
    ("Significant differences", f"{n_sig_bonf} of {len(tests)} after Bonferroni, {n_sig_bh} after Benjamini-Hochberg"),
    ("Redundancy", f"strongest pair {top_corr.iloc[0]['Feature 1']}-{top_corr.iloc[0]['Feature 2']} "
                   f"(r = {top_corr.iloc[0]['r']:.2f}); maximum VIF = {vif['VIF'].max():.2f}"),
    ("Baseline", f"logistic regression, all 23 predictors: CV ROC-AUC = "
                 f"{baseline.set_index('Feature set').loc['All 23 predictors', 'ROC-AUC mean']:.3f}; "
                 f"age + IDH1: {baseline.set_index('Feature set').loc['Age + IDH1', 'ROC-AUC mean']:.3f}"),
    ("Regression target", f"age mean {age.mean():.1f}, SD {age.std():.1f}; "
                          f"{results['n_age_significant_bh']} of {len(age_tests)} features related to age after BH; "
                          f"strongest: {age_tests.iloc[0]['Feature']} (median difference "
                          f"{age_tests.iloc[0]['Median difference']:+.1f} years)"),
]
summary = pd.DataFrame(summary_rows, columns=["Topic", "Finding"])
with pd.option_context("display.max_colwidth", None):
    display(summary)
results["summary"] = dict(summary_rows)
reporting.save_table_tex(summary, "eda_summary", column_format="lp{12cm}")
path = reporting.save_json(results, "02_eda")
print("Saved", path.relative_to(config.ROOT))
print("Figures:", sorted(p.name for p in config.FIGURES.glob("eda_*.png")))